# 01. Exploratory Data Analysis & Spatial Preprocessing
**Project:** Spatiotemporal Machine Learning & Multi-Horizon Deep Learning for Monsoon Precipitation Forecasting & Proactive Flood Early Warning in Nepal  
**Focus:** Satellite Precipitation (CHIRPS), Soil Moisture Reanalysis (ERA5-Land), and River Basin Topography.


In [ ]:
import os
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

print("Libraries imported successfully!")


## 1. Inspecting CHIRPS Satellite Precipitation (0.05° Grids)
Examining daily precipitation grids across the Nepal spatial bounding box: $[80.0^\circ	ext{E} - 88.2^\circ	ext{E}, 26.3^\circ	ext{N} - 30.5^\circ	ext{N}]$.


In [ ]:
chirps_path = "../data/raw/chirps/chirps_nepal_2022_monsoon.nc"
ds_chirps = xr.open_dataset(chirps_path)
print(ds_chirps)


In [ ]:
# Plotting spatial mean precipitation across Nepal on peak monsoon date
peak_idx = ds_chirps['precip'].mean(dim=['latitude', 'longitude']).argmax().item()
peak_date = str(ds_chirps['time'].values[peak_idx])[:10]

plt.figure(figsize=(10, 4.5))
ds_chirps['precip'].sel(time=peak_date).plot(cmap='YlGnBu')
plt.title(f"CHIRPS Spatial Precipitation Distribution on Peak Day: {peak_date} (mm/day)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()


## 2. Inspecting ECMWF ERA5-Land Volumetric Soil Moisture
Volumetric water layers for antecedent moisture representation:
- Layer 1 (`swvl1`): $0 - 7$ cm
- Layer 2 (`swvl2`): $7 - 28$ cm


In [ ]:
era5_path = "../data/raw/era5/era5_soil_moisture_2022_monsoon.nc"
ds_era5 = xr.open_dataset(era5_path)
print(ds_era5)


In [ ]:
# Comparing Top-Layer vs Deep-Layer Soil Moisture Temporal Variation
time_var = 'valid_time' if 'valid_time' in ds_era5.coords else 'time'
sm_l1 = ds_era5['swvl1'].mean(dim=['latitude', 'longitude']).to_series()
sm_l2 = ds_era5['swvl2'].mean(dim=['latitude', 'longitude']).to_series()

plt.figure(figsize=(11, 4))
plt.plot(sm_l1.index, sm_l1.values, label='Layer 1: 0-7 cm', color='#d95f02', lw=2)
plt.plot(sm_l2.index, sm_l2.values, label='Layer 2: 7-28 cm', color='#7570b3', lw=2, linestyle='--')
plt.title("Domain-Averaged Soil Moisture Progression (June-Sept 2022)")
plt.ylabel("Volumetric Soil Water (m³/m³)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
